# lab 3 Decision trees

## Activity 1 — Implement Entropy

Write an entropy(labels) function and test it on:

```python
labels1 = ["Pass", "Pass", "Pass", "Pass"]
labels2 = ["Pass", "Pass", "Fail", "Fail"]
labels3 = ["Pass", "Fail", "Pass", "Fail"]
```

Record the entropy of each set and explain why the values differ.

- Entropy: $H(Y) = -\sum p(y) \log_2 p(y)$
- Conditional Entropy: $H(Y\vert{}X) = \sum_{x} p(x) H(Y\vert{}X=x)$
- Mutual Information: $I(X; Y) = H(Y) - H(Y\vert{}X)$

In [9]:
import math


def entropy_func(a, b):
    total = a + b
    if total == 0:
        return 0.0

    p_a = a / total
    p_b = b / total

    # Avoid math.log2(0) by checking if probability > 0
    term_a = p_a * math.log2(p_a) if p_a > 0 else 0
    term_b = p_b * math.log2(p_b) if p_b > 0 else 0

    return -(term_a + term_b)



def entropy(labels):
    n = len(labels)
    if n == 0:
        return 0.0

    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1

    h = 0.0
    for c in counts.values():
        p = c / n
        h -= p * math.log2(p)
    return h + 0.0  # turns -0.0 into 0.0 for pure sets


# --- Test Cases ---
labels1 = ["Pass", "Pass", "Pass", "Pass"]
labels2 = ["Pass", "Fail", "Pass", "Fail"]
labels3 = ["Pass", "Fail", "Pass", "Fail"]

print(entropy(labels1))
print(entropy(labels2))
print(entropy(labels3))

0.0
1.0
1.0


## Activity 2 — Implement Mutual Information
Implement `conditional_entropy()` and `mutual_information()`. Calculate MI for all three features in the given dataset.

In [10]:
def conditional_entropy(feature_values, target_labels):
    """Calculates H(Y|X) - the conditional entropy of target Y given feature X."""
    total = len(target_labels)
    if total == 0:
        return 0.0

    # Group target labels by feature values
    groups = {}
    for x, y in zip(feature_values, target_labels):
        if x not in groups:
            groups[x] = []
        groups[x].append(y)

    # Calculate weighted sum of subset entropies: sum( p(x) * H(Y|X=x) )
    cond_ent = 0.0
    for x_val, y_subset in groups.items():
        weight = len(y_subset) / total
        cond_ent += weight * entropy(y_subset)

    return cond_ent


def mutual_information(feature_values, target_labels):
    """Calculates I(X; Y) = H(Y) - H(Y|X)"""
    h_y = entropy(target_labels)
    h_y_given_x = conditional_entropy(feature_values, target_labels)
    return h_y - h_y_given_x

## Activity 3 — Implement Prediction
Create a small Decision Tree manually and implement the prediction algorithm. Test at least three new students.

In [11]:
data = [
    {"Study": "High", "Attendance": "High", "Assignments": "Done",    "Result": "Pass"},
    {"Study": "High", "Attendance": "Low",  "Assignments": "Done",    "Result": "Pass"},
    {"Study": "High", "Attendance": "High", "Assignments": "Missing", "Result": "Pass"},
    {"Study": "Low",  "Attendance": "High", "Assignments": "Done",    "Result": "Pass"},
    {"Study": "Low",  "Attendance": "Low",  "Assignments": "Missing", "Result": "Fail"},
    {"Study": "Low",  "Attendance": "Low",  "Assignments": "Done",    "Result": "Fail"},
    {"Study": "Low",  "Attendance": "High", "Assignments": "Missing", "Result": "Fail"},
    {"Study": "High", "Attendance": "Low",  "Assignments": "Missing", "Result": "Fail"},
]
FEATURES = ["Study", "Attendance", "Assignments"]
TARGET = "Result"
labels = [r[TARGET] for r in data]

for f in FEATURES:
    print(f, round(mutual_information([r[f] for r in data], labels), 4))

def predict(tree, sample):
    while isinstance(tree, dict):
        value = sample.get(tree["feature"])
        tree = tree["children"].get(value, tree["default"])  # unseen value -> fallback
    return tree

manual_tree = {
    "feature": "Study", "default": "Pass",
    "children": {
        "High": "Pass",
        "Low": {
            "feature": "Attendance", "default": "Fail",
            "children": {"High": "Pass", "Low": "Fail"},
        },
    },
}

tests3 = [
    {"Study": "High", "Attendance": "Low",  "Assignments": "Done"},
    {"Study": "Low",  "Attendance": "High", "Assignments": "Missing"},
    {"Study": "Low",  "Attendance": "Low",  "Assignments": "Done"},
]
for s in tests3:
    print(s, "->", predict(manual_tree, s))


Study 0.1887
Attendance 0.1887
Assignments 0.1887
{'Study': 'High', 'Attendance': 'Low', 'Assignments': 'Done'} -> Pass
{'Study': 'Low', 'Attendance': 'High', 'Assignments': 'Missing'} -> Pass
{'Study': 'Low', 'Attendance': 'Low', 'Assignments': 'Done'} -> Fail


## Activity 4 — Implement Recursive Training
Implement `train()` using Mutual Information as the splitting criterion. Train a tree using the supplied dataset.


In [12]:
def majority(labels):
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    return max(counts, key=counts.get)

def train(rows, features, target):
    labels = [r[target] for r in rows]

    if len(set(labels)) == 1:          # pure node
        return labels[0]
    if not features:                   # nothing left to split on
        return majority(labels)

    gains = {f: mutual_information([r[f] for r in rows], labels) for f in features}
    best = max(gains, key=gains.get)
    if gains[best] == 0:               # no informative split
        return majority(labels)

    remaining = [f for f in features if f != best]
    children = {}
    for v in set(r[best] for r in rows):
        subset = [r for r in rows if r[best] == v]
        children[v] = train(subset, remaining, target)

    return {"feature": best, "default": majority(labels), "children": children}

tree = train(data, FEATURES, TARGET)

def show(t, indent=0):
    pad = "  " * indent
    if not isinstance(t, dict):
        print(f"{pad}-> {t}")
        return
    print(f"{pad}[{t['feature']}]")
    for v, sub in t["children"].items():
        print(f"{pad}  {v}:")
        show(sub, indent + 2)

show(tree)


[Study]
  High:
    [Attendance]
      High:
        -> Pass
      Low:
        [Assignments]
          Done:
            -> Pass
          Missing:
            -> Fail
  Low:
    [Attendance]
      High:
        [Assignments]
          Done:
            -> Pass
          Missing:
            -> Fail
      Low:
        -> Fail


## Activity 5 — Test the Trained Tree
Create at least five new student records and predict Pass/Fail for each record.

In [13]:
new_students = [
    {"Study": "High", "Attendance": "High", "Assignments": "Done"},
    {"Study": "High", "Attendance": "Low",  "Assignments": "Missing"},
    {"Study": "Low",  "Attendance": "High", "Assignments": "Done"},
    {"Study": "Low",  "Attendance": "Low",  "Assignments": "Missing"},
    {"Study": "High", "Attendance": "Low",  "Assignments": "Done"},
]
for s in new_students:
    print(s, "->", predict(tree, s))

{'Study': 'High', 'Attendance': 'High', 'Assignments': 'Done'} -> Pass
{'Study': 'High', 'Attendance': 'Low', 'Assignments': 'Missing'} -> Fail
{'Study': 'Low', 'Attendance': 'High', 'Assignments': 'Done'} -> Pass
{'Study': 'Low', 'Attendance': 'Low', 'Assignments': 'Missing'} -> Fail
{'Study': 'High', 'Attendance': 'Low', 'Assignments': 'Done'} -> Pass
